# Notebook 05 — Part 4 · Bank Failure Feature Pipeline (Stage 1 + Train-only Preprocessing)

**FDIC Bank Failure Early Warning · BFT65B Final Individual Report · 15 points**

**Mục tiêu:** (1) nhãn `fail_within_4q` được suy ra từ bảng sự kiện *retrospective* có kiểm chứng; (2) tạo các financial features theo nghiên cứu; (3) time-based Train / Validation / Test với **≥ 4 quý gap** và **≥ 50 test failed banks**; (4) fit preprocessing **chỉ trên Train**. Notebook đọc dữ liệu đã có, **không tải lại FDIC API** và **không ghi đè Part 1–3**.

**Thứ tự chạy:** chạy lần lượt từ trên xuống. `RUN_PREPROCESSOR=False` lúc đầu để kiểm chứng nhãn/split; sau khi báo cáo qua gates, bật True rồi chạy phần preprocess cuối. Không dùng `train_test_split(random)`; không tune bằng Test.

**Khoảng thời gian định trước, chưa tối ưu bằng Test:**

| Phase | Bank–quarter anchor | Explanation |
|---|---|---|
| Train | 1984Q1–2005Q4 | 1980s–1990s distress plus pre-crisis banks |
| Purge | 2006Q1–2006Q4 | 4 quarters between Train and Validation |
| Validation | 2007Q1–2011Q4 | Development-only model selection, includes global financial crisis |
| Purge | 2012Q1–2012Q4 | 4 quarters between Development and Test |
| **Test** | **2013Q1–2025Q2** | Held-out post-crisis/recent period; ≥50 distinct failed banks is an automatic gate |
| Outside split | No post-2025Q2 modeling anchors; only 2006 and 2012 are intentionally excluded as purge gaps |

**Target horizon:** failure in `(q, q+4 quarters]` (the following four calendar quarters). **Feature timing:** `FEATURE_LAG_QUARTERS=1`, so predictors at anchor q come from q−1 and earlier. Note this is a *conservative proxy*, not proof of actual Call Report release date; official report vintages/restatements are not available here. Sensitivity with two-quarter lag is recommended.

**Important:** The study label targets *FDIC failure*, not all forms of distress; assisted merger codes 213 remain unresolved and are **not** silently relabeled as nonfailure. The source's retrospective outcomes are used only for label verification and censoring, never included as predictors.

**Patch:** FIXED version — resolves `LazyFrame.height`, schema warnings, and mixed LazyFrame/DataFrame join in feature matching.


## Academic foundations & field-definition evidence

The feature **groups** follow established banking surveillance, **not** a claim that each derived ratio is reproduced exactly from the papers.

1. **FDIC SCOR** — Collier, Forbush, Nuxoll & O'Keefe (2003), *The SCOR System of Off-Site Monitoring*, FDIC Banking Review: traditional CAMELS-related financial information and lagged financial indicators; [FDIC Banking Review archive](https://archive.fdic.gov/view/fdic/9808/fdic_9808_DS1.pdf).
2. **Cole & White (2012)** — *Déjà Vu All Over Again: The Causes of U.S. Commercial Bank Failures This Time Around*, Journal of Financial Services Research, 42, 5–29: CAMELS proxies and commercial real-estate exposures; [NYU paper](https://www.stern.nyu.edu/sites/default/files/assets/documents/con_039550.pdf); [Yale archive](https://elischolar.library.yale.edu/ypfs-documents/1141/).
3. **Betz, Oprică, Peltonen & Sarlin (2014)** — *Predicting distress in European banks*, Journal of Banking & Finance, 45, 225–241: bank-specific CAMELS and macro-financial vulnerability signals; **their target includes broader distress**, so their findings motivate predictor groups only, not the FDIC failure label; [DOI](https://doi.org/10.1016/j.jbankfin.2013.11.041).
4. **FDIC BankFind Suite documentation** — official Financials, Failures, Institutions, and History field definitions, bank-level identifiers, and event semantics: https://api.fdic.gov/banks/docs/ . Field definitions come from the supplied `risview_properties.yaml` / data dictionary; this notebook provides a formula catalog for proxies, not FDIC-defined official ratios.

**Modeling caution:** `RBC1AAJ` is FDIC's *leverage ratio – PCA*; `RBC1RWAJ` is *Tier 1 risk-based capital ratio – PCA*. They should **not** be interpreted as interchangeable. `ROA`, `ROE`, `NIMY` already appear as percent-valued ratios; this notebook does not multiply them by 100. `NETINC` may be period/YTD-related and is not independently normalized here. `NCLNLS`, `ORE`, and `LNATRES` are checked for negative values before building any nonnegative ratio. Missing historic availability is retained and profiled.

## 0 · Setup & immutable input paths

In [ ]:
%pip -q install "polars>=1.20,<2" "pandas>=2.2,<3" "pyarrow>=17,<25" "scikit-learn>=1.4,<2" "joblib>=1.3,<2"

In [ ]:
import os, json, sys, platform, zipfile, tempfile, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq

ROOT_OVERRIDE = os.environ.get('FDIC_PROJECT_ROOT')
if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')

CLEAN = ROOT / 'data_processed/clean_panel_v1'
HISTORY = ROOT / 'data_processed/part2_history_exit_reference_v1/bank_exit_outcomes_with_history.parquet'
EVENTS = ROOT / 'data_processed/part2_event_reference_v1/failure_event_alignment.parquet'
OUT = ROOT / 'outputs/part4/feature_pipeline_stage1_v1'
MODEL_DATA = ROOT / 'data_processed/feature_panel_v1'

FEATURE_LAG_QUARTERS = 1             # q−1 at prediction anchor q; run 2 as later sensitivity
RUN_PREPROCESSOR = True             # switch to True only after validating target and split
EXPECTED_FINANCIAL_ROWS = 1_682_616
LAST_OBSERVABLE_QUARTER = '2026Q2'
LAST_MODEL_ANCHOR_QUARTER = '2025Q2'
SPLIT_BOUNDS = {
    'TRAIN': ('1984Q1','2005Q4'),
    'VALIDATION': ('2007Q1','2011Q4'),
    'TEST': ('2013Q1','2025Q2'),
}
assert FEATURE_LAG_QUARTERS in (1,2)
assert CLEAN.exists() and HISTORY.is_file() and EVENTS.is_file(), (
    'Missing clean panel or event references; check your Part 2 Drive outputs',CLEAN,HISTORY,EVENTS)
files=sorted(CLEAN.glob('YEAR=*/data.parquet'))
assert len(files)==43, f'Expected 43 clean yearly Parquet files, found {len(files)}'
assert sum(pq.ParquetFile(f).metadata.num_rows for f in files)==EXPECTED_FINANCIAL_ROWS
OUT.mkdir(parents=True,exist_ok=True)
MODEL_DATA.mkdir(parents=True,exist_ok=True)
print('INPUT OK | clean panel years:',len(files),'| total rows:',EXPECTED_FINANCIAL_ROWS)
print('Output:',OUT)
print('Polars:',pl.__version__,'pandas:',pd.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INPUT OK | clean panel years: 43 | total rows: 1682616
Output: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part4/feature_pipeline_stage1_v1
Polars: 1.35.2 pandas: 2.2.3


## 1 · Calendar-based split design and immutable guards

`q_ord = year × 4 + quarter − 1`. Four missing calendar quarters separate the *end of Train* from the *start of Validation*, and likewise the end of Validation from the start of Test. We will additionally ensure **last Train label window** ends before **first Validation anchor**, and **last Validation label window** ends before **first Test anchor**. Both gaps are determined in advance and **not adjusted based on observed Test model performance**.

In [ ]:
def qord(q):
    p=pd.Period(q, freq='Q')
    return int(p.year*4 + (p.quarter-1))

def qname(ordinal):
    year, offset = divmod(int(ordinal), 4)
    return f'{year}Q{offset+1}'

BOUNDS = {name:(qord(lo),qord(hi)) for name,(lo,hi) in SPLIT_BOUNDS.items()}
assert BOUNDS['TRAIN'][1]+5 <= BOUNDS['VALIDATION'][0]
assert BOUNDS['VALIDATION'][1]+5 <= BOUNDS['TEST'][0]
assert BOUNDS['TRAIN'][1]+4 < BOUNDS['VALIDATION'][0]
assert BOUNDS['VALIDATION'][1]+4 < BOUNDS['TEST'][0]
assert qord(LAST_MODEL_ANCHOR_QUARTER)+4 == qord(LAST_OBSERVABLE_QUARTER)
print('Pre-registered split:',BOUNDS)
print('Target ends for last allowable model anchor:',qname(qord(LAST_MODEL_ANCHOR_QUARTER)+4))

Pre-registered split: {'TRAIN': (7936, 8023), 'VALIDATION': (8028, 8047), 'TEST': (8052, 8101)}
Target ends for last allowable model anchor: 2026Q2


## 2 · Source schema and feature catalog (candidate set, not post-hoc selection)

Formulas use only **past accounting information**. All financial amounts use identical FDIC source units. Ratios are fractional unless suffixed `_PCT` (source percent). No division by `0` or negative denominator is allowed. A negative bank equity balance is *not* automatically invalidated. Fields missing entirely from the clean panel cause candidate variables to be skipped **with an audit record**, rather than fabricated.

In [ ]:
scan = pl.scan_parquet([str(f) for f in files], hive_partitioning=False)
SCHEMA = set(scan.collect_schema().names())
assert {'CERT','REPDTE','ASSET','EQ'} <= SCHEMA

# group, feature, dependencies, formula/policy, research grounding
CATALOG = [
 ('Capital','F_EQUITY_ASSET',['EQ','ASSET'],'EQ / ASSET','CAMELS; Cole & White (2012)'),
 ('Capital','F_PCA_LEVERAGE_PCT',['RBC1AAJ'],'RBC1AAJ (already %)','FDIC financial field; SCOR'),
 ('Capital','F_TIER1_RBC_PCT',['RBC1RWAJ'],'RBC1RWAJ (already %)','FDIC financial field; SCOR'),
 ('Asset quality','F_NPL_LOANS',['NCLNLS','LNLSGR'],'max_valid(NCLNLS) / LNLSGR','CAMELS; Cole & White (2012)'),
 ('Asset quality','F_NONACCRUAL_ASSET',['NAASSET','ASSET'],'NAASSET / ASSET','CAMELS; FDIC SCOR'),
 ('Asset quality','F_PASTDUE90_ASSET',['P9ASSET','ASSET'],'P9ASSET / ASSET','CAMELS; FDIC SCOR'),
 ('Asset quality','F_NET_CHARGEOFFS_LOANS',['NTLNLS','LNLSGR'],'NTLNLS / LNLSGR (signed)','CAMELS; FDIC SCOR'),
 ('Asset quality','F_TEXAS_PROXY',['NCLNLS','ORE','EQ','INTAN','LNATRES'],'(NCLNLS + ORE) / (EQ - INTAN + LNATRES); proxy','Research-style asset quality; not an official FDIC measure'),
 ('Earnings','F_ROA_PCT',['ROA'],'ROA (FDIC already %)','CAMELS; Cole & White (2012)'),
 ('Earnings','F_ROE_PCT',['ROE'],'ROE (FDIC already %)','CAMELS'),
 ('Earnings','F_NIM_PCT',['NIMY'],'NIMY (already %)','CAMELS; FDIC SCOR'),
 ('Liquidity','F_CASH_ASSET',['CHBAL','ASSET'],'CHBAL / ASSET','CAMELS; Betz et al. (2014)'),
 ('Liquidity','F_SECURITIES_ASSET',['SC','ASSET'],'SC / ASSET','CAMELS; Betz et al. (2014)'),
 ('Funding','F_DEPOSITS_ASSET',['DEP','ASSET'],'DEP / ASSET','CAMELS; FDIC SCOR'),
 ('Funding','F_COREDEP_SHARE',['COREDEP','DEP'],'COREDEP / DEP','CAMELS; FDIC SCOR'),
 ('Funding','F_UNINSURED_SHARE',['DEPUNINS','DEP'],'DEPUNINS / DEP','Funding-run vulnerability (availability varies by year)'),
 ('Funding','F_LOANS_DEPOSITS',['LNLSGR','DEP'],'LNLSGR / DEP','Liquidity/funding proxy'),
 ('Concentration','F_RE_LOAN_SHARE',['LNRE','LNLSGR'],'LNRE / LNLSGR','Cole & White (2012)'),
 ('Concentration','F_CRE_LOAN_SHARE',['LNCOMRE','LNLSGR'],'LNCOMRE / LNLSGR','Cole & White (2012)'),
 ('Concentration','F_CONSTRUCTION_LOAN_SHARE',['LNRECONS','LNLSGR'],'LNRECONS / LNLSGR','Cole & White (2012)'),
 ('Size','F_LOG_ASSET',['ASSET'],'log(ASSET) for ASSET>0','Common bank-level control'),
 ('Dynamics','F_ASSET_GROWTH_QOQ',['ASSET'],'ASSET_t / ASSET_(t−1) − 1; strict consecutive quarter','Lagged condition; FDIC SCOR'),
 ('Dynamics','F_DEP_GROWTH_QOQ',['DEP'],'DEP_t / DEP_(t−1) − 1; strict consecutive quarter','Lagged condition; FDIC SCOR'),
 ('Dynamics','F_ROA_CHANGE_4Q',['ROA'],'ROA_t − ROA_(t−4); exact 4-quarter offset','CAMELS dynamics'),
 ('Dynamics','F_NPL_CHANGE_4Q',['NCLNLS','LNLSGR'],'NPL ratio_t − ratio_(t−4); exact offset','Asset-quality dynamics'),
 ('Dynamics','F_ROA_ROLL4',['ROA'],'mean 4 consecutive quarterly ROA values (source through t)','Earnings persistence'),
]
feature_catalog = pd.DataFrame(CATALOG,columns=['group','feature','source_fields','formula','reference_basis'])
feature_catalog['available'] = feature_catalog.source_fields.apply(lambda fs: set(fs)<=SCHEMA)
feature_catalog['missing_columns'] = feature_catalog.source_fields.apply(lambda fs: ','.join(sorted(set(fs)-SCHEMA)))
SELECTABLE = feature_catalog[feature_catalog.available].feature.tolist()
assert len(SELECTABLE)>=10, 'Too few available features: inspect Financials schema'
feature_catalog.to_csv(OUT/'candidate_feature_catalog.csv',index=False)
display(feature_catalog[['group','feature','formula','available','missing_columns']])
print('Candidates available:',len(SELECTABLE),'of',len(CATALOG))

,group,feature,formula,available,missing_columns
0,Capital,F_EQUITY_ASSET,EQ / ASSET,True,
1,Capital,F_PCA_LEVERAGE_PCT,RBC1AAJ (already %),True,
2,Capital,F_TIER1_RBC_PCT,RBC1RWAJ (already %),True,
3,Asset quality,F_NPL_LOANS,max_valid(NCLNLS) / LNLSGR,True,
4,Asset quality,F_NONACCRUAL_ASSET,NAASSET / ASSET,True,
5,Asset quality,F_PASTDUE90_ASSET,P9ASSET / ASSET,True,
6,Asset quality,F_NET_CHARGEOFFS_LOANS,NTLNLS / LNLSGR (signed),True,
7,Asset quality,F_TEXAS_PROXY,(NCLNLS + ORE) / (EQ - INTAN + LNATRES); proxy,True,
8,Earnings,F_ROA_PCT,ROA (FDIC already %),True,
9,Earnings,F_ROE_PCT,ROE (FDIC already %),True,


Candidates available: 26 of 26


## 3 · Polars deterministic feature construction, with strict bank-quarter chronology

The timestamp at anchor q is a **prediction anchor**, not a source reporting date. Financial predictors are calculated at source quarter s and then attached only when `s = q − FEATURE_LAG_QUARTERS`. No latest-observation carry-forward is permitted. Prior-quarter growth / 4Q differences require exact calendar continuity; data gaps produce null, not invented lag values. Ratio component guards are applied to `NCLNLS`, `ORE`, `LNATRES` etc. in addition to the 03B `*_QA_VALID` columns.

**FIXED NOTE (2026-11-17):** This pipeline collects the final LazyFrame exactly once before eager assertions, obtains the lazy schema via `collect_schema().names()`, and computes `FEATURE_SOURCE_MATCHED` during the original exact bank–quarter join. Do not call `Run` on Section 5 alone before rerunning Section 3.


In [ ]:
# Column-pruned scan. No retrospective event field can enter this source.
required = set(['CERT','REPDTE'])
for fs in feature_catalog.loc[feature_catalog.available,'source_fields']:
    required.update(fs)
cols=sorted(required)
raw=scan.select(cols)

BAD_FEATURE_INPUTS = {
    'FAILDATE','FAILDATE_DT','ACTIVE','ENDEFYMD','RETROSPECTIVE_EXIT_CLASS',
    'REVISED_EXIT_CLASS_HISTORY','HISTORY_EFFDATE','ALIGNMENT_CONFIRMED_FOR_MODEL_LABEL'
}
assert not (set(cols) & BAD_FEATURE_INPUTS)

# Ensure numeric columns behave uniformly; raw source values are not overwritten in Part 2.
raw=raw.with_columns([
    pl.col(c).cast(pl.Float64,strict=False).alias(c)
    for c in cols if c not in ('CERT','REPDTE')
])
raw=raw.with_columns([
    (pl.col('REPDTE').str.slice(0,4).cast(pl.Int32)*4 +
     pl.col('REPDTE').str.slice(4,2).cast(pl.Int32)//3-1).alias('Q_ORD')
])

# ratio helpers; force both operands to be finite and require positive denominator.
def safe_div(numerator, denominator):
    n = numerator.cast(pl.Float64)
    d = denominator.cast(pl.Float64)
    ok=n.is_finite().fill_null(False) & d.is_finite().fill_null(False) & (d>0).fill_null(False)
    return pl.when(ok).then(n/d).otherwise(None)

def positive_stock(field):
    x=pl.col(field).cast(pl.Float64)
    return pl.when((x>=0).fill_null(False) & x.is_finite().fill_null(False)).then(x).otherwise(None)

exprs={
 'F_EQUITY_ASSET': safe_div(pl.col('EQ'),positive_stock('ASSET')),
 'F_PCA_LEVERAGE_PCT': pl.col('RBC1AAJ') if 'RBC1AAJ' in SCHEMA else pl.lit(None),
 'F_TIER1_RBC_PCT': pl.col('RBC1RWAJ') if 'RBC1RWAJ' in SCHEMA else pl.lit(None),
 'F_NPL_LOANS': safe_div(positive_stock('NCLNLS'),positive_stock('LNLSGR')) if {'NCLNLS','LNLSGR'}<=SCHEMA else pl.lit(None),
 'F_NONACCRUAL_ASSET': safe_div(positive_stock('NAASSET'),positive_stock('ASSET')) if 'NAASSET' in SCHEMA else pl.lit(None),
 'F_PASTDUE90_ASSET': safe_div(positive_stock('P9ASSET'),positive_stock('ASSET')) if 'P9ASSET' in SCHEMA else pl.lit(None),
 'F_NET_CHARGEOFFS_LOANS': safe_div(pl.col('NTLNLS'),positive_stock('LNLSGR')) if {'NTLNLS','LNLSGR'}<=SCHEMA else pl.lit(None),
 'F_TEXAS_PROXY': safe_div(positive_stock('NCLNLS')+positive_stock('ORE'), pl.col('EQ')-positive_stock('INTAN')+positive_stock('LNATRES')) if {'NCLNLS','ORE','EQ','INTAN','LNATRES'}<=SCHEMA else pl.lit(None),
 'F_ROA_PCT': pl.col('ROA') if 'ROA' in SCHEMA else pl.lit(None),
 'F_ROE_PCT': pl.col('ROE') if 'ROE' in SCHEMA else pl.lit(None),
 'F_NIM_PCT': pl.col('NIMY') if 'NIMY' in SCHEMA else pl.lit(None),
 'F_CASH_ASSET': safe_div(positive_stock('CHBAL'),positive_stock('ASSET')) if 'CHBAL' in SCHEMA else pl.lit(None),
 'F_SECURITIES_ASSET': safe_div(positive_stock('SC'),positive_stock('ASSET')) if 'SC' in SCHEMA else pl.lit(None),
 'F_DEPOSITS_ASSET': safe_div(positive_stock('DEP'),positive_stock('ASSET')) if 'DEP' in SCHEMA else pl.lit(None),
 'F_COREDEP_SHARE': safe_div(positive_stock('COREDEP'),positive_stock('DEP')) if {'COREDEP','DEP'}<=SCHEMA else pl.lit(None),
 'F_UNINSURED_SHARE': safe_div(positive_stock('DEPUNINS'),positive_stock('DEP')) if {'DEPUNINS','DEP'}<=SCHEMA else pl.lit(None),
 'F_LOANS_DEPOSITS': safe_div(positive_stock('LNLSGR'),positive_stock('DEP')) if {'LNLSGR','DEP'}<=SCHEMA else pl.lit(None),
 'F_RE_LOAN_SHARE': safe_div(positive_stock('LNRE'),positive_stock('LNLSGR')) if {'LNRE','LNLSGR'}<=SCHEMA else pl.lit(None),
 'F_CRE_LOAN_SHARE': safe_div(positive_stock('LNCOMRE'),positive_stock('LNLSGR')) if {'LNCOMRE','LNLSGR'}<=SCHEMA else pl.lit(None),
 'F_CONSTRUCTION_LOAN_SHARE': safe_div(positive_stock('LNRECONS'),positive_stock('LNLSGR')) if {'LNRECONS','LNLSGR'}<=SCHEMA else pl.lit(None),
 'F_LOG_ASSET': pl.when((pl.col('ASSET')>0).fill_null(False)).then(pl.col('ASSET').log()).otherwise(None),
}

base=raw.select(['CERT','REPDTE','Q_ORD']+[c for c in cols if c not in ('CERT','REPDTE')])
base=base.sort(['CERT','Q_ORD']).with_columns([
    exprs[f].cast(pl.Float64).alias(f)
    for f in SELECTABLE if f in exprs
])

# Dynamic features reference only their own bank's historical source quarters.
base=base.with_columns([
    pl.col('Q_ORD').shift(1).over('CERT').alias('_PREV_Q'),
    pl.col('Q_ORD').shift(4).over('CERT').alias('_Q4_AGO'),
    pl.col('ASSET').shift(1).over('CERT').alias('_PREV_ASSET'),
    pl.col('DEP').shift(1).over('CERT').alias('_PREV_DEP') if 'DEP' in SCHEMA else pl.lit(None).alias('_PREV_DEP'),
    pl.col('ROA').shift(4).over('CERT').alias('_ROA_Q4_AGO') if 'ROA' in SCHEMA else pl.lit(None).alias('_ROA_Q4_AGO'),
    pl.col('F_NPL_LOANS').shift(4).over('CERT').alias('_NPL_Q4_AGO') if 'F_NPL_LOANS' in SELECTABLE else pl.lit(None).alias('_NPL_Q4_AGO'),
])
# Rolling window uses the immediately preceding 4 source rows and requires consecutive quarters.
base=base.with_columns([
    pl.col('Q_ORD').shift(3).over('CERT').alias('_Q3_AGO'),
    pl.col('ROA').rolling_mean(window_size=4,min_samples=4).over('CERT').alias('_ROA_ROLL4') if 'ROA' in SCHEMA else pl.lit(None).alias('_ROA_ROLL4'),
])
new_exprs={
 'F_ASSET_GROWTH_QOQ': pl.when(pl.col('Q_ORD')-pl.col('_PREV_Q')==1).then(safe_div(pl.col('ASSET'),positive_stock('_PREV_ASSET'))-1).otherwise(None),
 'F_DEP_GROWTH_QOQ': pl.when(pl.col('Q_ORD')-pl.col('_PREV_Q')==1).then(safe_div(pl.col('DEP'),positive_stock('_PREV_DEP'))-1).otherwise(None) if 'DEP' in SCHEMA else pl.lit(None),
 'F_ROA_CHANGE_4Q': pl.when(pl.col('Q_ORD')-pl.col('_Q4_AGO')==4).then(pl.col('ROA')-pl.col('_ROA_Q4_AGO')).otherwise(None) if 'ROA' in SCHEMA else pl.lit(None),
 'F_NPL_CHANGE_4Q': pl.when(pl.col('Q_ORD')-pl.col('_Q4_AGO')==4).then(pl.col('F_NPL_LOANS')-pl.col('_NPL_Q4_AGO')).otherwise(None) if 'F_NPL_LOANS' in SELECTABLE else pl.lit(None),
 'F_ROA_ROLL4': pl.when(pl.col('Q_ORD')-pl.col('_Q3_AGO')==3).then(pl.col('_ROA_ROLL4')).otherwise(None) if 'ROA' in SCHEMA else pl.lit(None),
}
base=base.with_columns([new_exprs[f].cast(pl.Float64).alias(f) for f in SELECTABLE if f in new_exprs])
base_schema = set(base.collect_schema().names())  # resolve lazy schema once
for f in SELECTABLE:
    assert f in base_schema, f

# Attach an explicit source-row existence flag BEFORE the lazy join.
# A null financial ratio is NOT proof that a source report is absent.
source_features = (
    base.select(['CERT', 'Q_ORD'] + SELECTABLE)
        .rename({'Q_ORD': 'FEATURE_SOURCE_Q_ORD'})
        .with_columns(pl.lit(True).alias('FEATURE_SOURCE_MATCHED'))
)
anchors = base.select(['CERT', 'REPDTE', 'Q_ORD'])
feature_panel = (
    anchors.with_columns(
        (pl.col('Q_ORD') - FEATURE_LAG_QUARTERS).alias('FEATURE_SOURCE_Q_ORD')
    )
    .join(source_features, on=['CERT', 'FEATURE_SOURCE_Q_ORD'], how='left')
    .with_columns(pl.col('FEATURE_SOURCE_MATCHED').fill_null(False))
    .sort(['CERT', 'Q_ORD'])
    .collect()  # LazyFrame -> eager DataFrame: .height/.shape/.item() work now
)
assert feature_panel.height == EXPECTED_FINANCIAL_ROWS
assert feature_panel.select(pl.struct(['CERT', 'REPDTE']).n_unique()).item() == EXPECTED_FINANCIAL_ROWS
assert not (BAD_FEATURE_INPUTS & set(feature_panel.columns))
print('Built lag-safe candidate panel:', feature_panel.shape)
print('Exact prior-quarter source rows matched:',
      feature_panel.select(pl.col('FEATURE_SOURCE_MATCHED').sum()).item())

Built lag-safe candidate panel: (1682616, 31)
Exact prior-quarter source rows matched: 1658519


## 4 · Load retrospective **outcome-only** event references; no outcome fields in X

In [ ]:
history = pl.read_parquet(str(HISTORY)).to_pandas()
events = pl.read_parquet(str(EVENTS)).to_pandas()
assert history.CERT.is_unique and len(history)==24064
required_history={'CERT','LAST_FINANCIAL_QUARTER','REVISED_EXIT_CLASS_HISTORY','HISTORY_EFFDATE'}
required_events={'CERT','FAILDATE_DT','ALIGNMENT_CONFIRMED_FOR_MODEL_LABEL','WITHIN_STUDY','EVENT_ALIGNMENT_STATUS'}
assert required_history<=set(history.columns)
assert required_events<=set(events.columns)

def to_qord_date(series):
    s=pd.to_datetime(series,errors='coerce')
    return (s.dt.year*4+s.dt.quarter-1).astype('Float64')

def to_qord_yyyymmdd(series):
    s=series.astype('string').str.replace(r'\.0$', '', regex=True)
    dt=pd.to_datetime(s,format='%Y%m%d',errors='coerce')
    return (dt.dt.year*4+dt.dt.quarter-1).astype('Float64')

history['LAST_Q'] = to_qord_yyyymmdd(history['LAST_FINANCIAL_QUARTER'])
history['MERGER_Q'] = to_qord_date(history['HISTORY_EFFDATE']).where(
    history['REVISED_EXIT_CLASS_HISTORY'].eq('MERGER_VERIFIED_HISTORY'))

# Only 03B-aligned failures are positive labels; keep ambiguous failures for censoring.
events['CERT']=pd.to_numeric(events.CERT,errors='coerce')
events=events[events.CERT.notna() & events.WITHIN_STUDY.fillna(False)].copy()
events['CERT']=events.CERT.astype('int64')
events['EVENT_Q']=to_qord_date(events['FAILDATE_DT'])
confirmed=events[events['ALIGNMENT_CONFIRMED_FOR_MODEL_LABEL'].fillna(False)].copy()
assert not confirmed.CERT.duplicated().any(), 'Multiple aligned failures for a CERT must be reviewed'
confirmed_map=confirmed.set_index('CERT')['EVENT_Q']
review=events[~events['ALIGNMENT_CONFIRMED_FOR_MODEL_LABEL'].fillna(False)].copy()
review_min=review.groupby('CERT')['EVENT_Q'].min()
multiple_certs=set(events.groupby('CERT').size().loc[lambda x: x>1].index)

outcome = history[['CERT','REVISED_EXIT_CLASS_HISTORY','LAST_Q','MERGER_Q']].copy()
outcome['FAIL_Q']=outcome.CERT.map(confirmed_map).astype('Float64')
outcome['REVIEW_MIN_Q']=outcome.CERT.map(review_min).astype('Float64')
outcome['MULTIPLE_FAILURE_EVENTS_CERT']=outcome.CERT.isin(multiple_certs)
assert outcome.CERT.is_unique
print('Events in study:',len(events),'| aligned:',len(confirmed),'| ambiguous:',len(review))
print('History classes:\n',history.REVISED_EXIT_CLASS_HISTORY.value_counts())

Events in study: 3273 | aligned: 2768 | ambiguous: 505
History classes:
 REVISED_EXIT_CLASS_HISTORY
MERGER_VERIFIED_HISTORY      14346
RIGHT_CENSORED_AT_END         4307
FAILURE_CONFIRMED_ALIGNED     2768
OTHER_EXIT_DATE_SUPPORTED     2038
FAILURE_EVENT_LINK_REVIEW      448
EXIT_UNVERIFIED                136
OTHER_EXIT_HISTORY_REVIEW       21
Name: count, dtype: int64


## 5 · Assign 4-quarter outcome labels **before** any model fitting

A zero label must be justified by observed survival through q+4, a verified merger **after** q+4, or an aligned failure **after** q+4. An exit **within** the horizon without confirmed FDIC failure is **unknown/censored**, **not** 0. Only a unique, aligned FDIC failure can yield 1. Ambiguous failure links are conservatively excluded if they overlap the horizon; multiple-event CERTs are excluded entirely. No assumption that an inactive snapshot or a missing History row proves a non-failure.

In [ ]:
def label_bank_quarters(panel: pd.DataFrame, outcomes: pd.DataFrame,
                        final_model_q: int) -> pd.DataFrame:
    assert {'CERT','Q_ORD','FEATURE_SOURCE_Q_ORD'}<=set(panel.columns)
    assert outcomes.CERT.is_unique
    a=panel.merge(outcomes,on='CERT',how='left',validate='many_to_one')
    assert len(a)==len(panel) and a['REVISED_EXIT_CLASS_HISTORY'].notna().all()
    q=a.Q_ORD.astype('int64')
    horizon=q+4
    failure=a['FAIL_Q']
    review=a['REVIEW_MIN_Q']
    bank_class=a['REVISED_EXIT_CLASS_HISTORY']
    ambiguous=bank_class.eq('FAILURE_EVENT_LINK_REVIEW') | a['MULTIPLE_FAILURE_EVENTS_CERT'].fillna(False)
    review_overlap=review.notna() & (review<=horizon)
    # Cases with an ambiguous event on/before q+4 can never be labeled 0.
    review_block = review_overlap | a['MULTIPLE_FAILURE_EVENTS_CERT'].fillna(False)
    has_confirmed=bank_class.eq('FAILURE_CONFIRMED_ALIGNED') & failure.notna()
    positive = has_confirmed & (failure>q) & (failure<=horizon) & ~review_block
    at_risk = (~has_confirmed | (failure>q))
    # Evidence of survival to horizon (uses future information *solely* for labeling):
    follows_from_reports = a['LAST_Q'].notna() & (a['LAST_Q']>=horizon)
    follows_from_later_failure = has_confirmed & (failure>horizon)
    follows_from_later_merger = bank_class.eq('MERGER_VERIFIED_HISTORY') & a['MERGER_Q'].notna() & (a['MERGER_Q']>horizon)
    safe_negative=(follows_from_reports | follows_from_later_failure | follows_from_later_merger) & at_risk & ~review_block & ~positive
    a['fail_within_4q']=pd.Series(pd.array([pd.NA]*len(a),dtype='Int8'))
    a.loc[safe_negative,'fail_within_4q']=0
    a.loc[positive,'fail_within_4q']=1
    a['LABEL_STATUS']='UNKNOWN_CENSORED_OR_UNVERIFIED'
    a.loc[safe_negative,'LABEL_STATUS']='OBSERVED_NO_FAILURE_4Q'
    a.loc[positive,'LABEL_STATUS']='CONFIRMED_FAILURE_WITHIN_4Q'
    a.loc[review_block,'LABEL_STATUS']='AMBIGUOUS_FAILURE_EVENT_LINK'
    a.loc[q>final_model_q,'LABEL_STATUS']='INSUFFICIENT_FOLLOWUP_AT_DATA_END'
    a.loc[q>final_model_q,'fail_within_4q']=pd.NA
    # Source-report availability is checked separately via an exact (CERT, q-lag) join.
    return a

# Unit tests for target logic, independent of the FDIC dataset.
# These cases check event horizon, merger censoring, review, and last-quarter truncation.
toy=pd.DataFrame([
  {'CERT':1,'Q_ORD':qord('2008Q2'),'FEATURE_SOURCE_Q_ORD':qord('2008Q1')},
  {'CERT':1,'Q_ORD':qord('2007Q1'),'FEATURE_SOURCE_Q_ORD':qord('2006Q4')},
  {'CERT':2,'Q_ORD':qord('2008Q1'),'FEATURE_SOURCE_Q_ORD':qord('2007Q4')},
  {'CERT':2,'Q_ORD':qord('2007Q1'),'FEATURE_SOURCE_Q_ORD':qord('2006Q4')},
  {'CERT':3,'Q_ORD':qord('2008Q1'),'FEATURE_SOURCE_Q_ORD':qord('2007Q4')},
  {'CERT':3,'Q_ORD':qord('2007Q1'),'FEATURE_SOURCE_Q_ORD':qord('2006Q4')},
  {'CERT':4,'Q_ORD':qord('2009Q1'),'FEATURE_SOURCE_Q_ORD':qord('2008Q4')},
  {'CERT':4,'Q_ORD':qord('2007Q1'),'FEATURE_SOURCE_Q_ORD':qord('2006Q4')},
  {'CERT':5,'Q_ORD':qord('2025Q3'),'FEATURE_SOURCE_Q_ORD':qord('2025Q2')},
])
toy_out=pd.DataFrame([
  {'CERT':1,'REVISED_EXIT_CLASS_HISTORY':'FAILURE_CONFIRMED_ALIGNED','LAST_Q':qord('2009Q1'),'MERGER_Q':pd.NA,'FAIL_Q':qord('2009Q2'),'REVIEW_MIN_Q':pd.NA,'MULTIPLE_FAILURE_EVENTS_CERT':False},
  {'CERT':2,'REVISED_EXIT_CLASS_HISTORY':'MERGER_VERIFIED_HISTORY','LAST_Q':qord('2008Q2'),'MERGER_Q':qord('2008Q3'),'FAIL_Q':pd.NA,'REVIEW_MIN_Q':pd.NA,'MULTIPLE_FAILURE_EVENTS_CERT':False},
  {'CERT':3,'REVISED_EXIT_CLASS_HISTORY':'EXIT_UNVERIFIED','LAST_Q':qord('2008Q2'),'MERGER_Q':pd.NA,'FAIL_Q':pd.NA,'REVIEW_MIN_Q':pd.NA,'MULTIPLE_FAILURE_EVENTS_CERT':False},
  {'CERT':4,'REVISED_EXIT_CLASS_HISTORY':'FAILURE_EVENT_LINK_REVIEW','LAST_Q':qord('2010Q2'),'MERGER_Q':pd.NA,'FAIL_Q':pd.NA,'REVIEW_MIN_Q':qord('2010Q1'),'MULTIPLE_FAILURE_EVENTS_CERT':False},
  {'CERT':5,'REVISED_EXIT_CLASS_HISTORY':'RIGHT_CENSORED_AT_END','LAST_Q':qord('2026Q2'),'MERGER_Q':pd.NA,'FAIL_Q':pd.NA,'REVIEW_MIN_Q':pd.NA,'MULTIPLE_FAILURE_EVENTS_CERT':False},
])
toy['F_EQUITY_ASSET']=0.05
toy_result=label_bank_quarters(toy,toy_out,qord('2025Q2'))
expected=[1,0,None,0,None,0,None,0,None]
for actual,want in zip(toy_result.fail_within_4q.tolist(),expected):
    assert (pd.isna(actual) if want is None else int(actual)==want), (actual,want)
print('Synthetic target/exit/embargo edge cases: 9/9 PASS')

# Collect only model candidates and time/key metadata into pandas.
# FEATURE_SOURCE_MATCHED was computed from the exact source-key join in Section 3;
# avoid another 1.68M-row join, and do NOT infer presence from non-null ratios.
base_label = feature_panel.select(
    ['CERT', 'REPDTE', 'Q_ORD', 'FEATURE_SOURCE_Q_ORD', 'FEATURE_SOURCE_MATCHED']
    + SELECTABLE
).to_pandas()
base_label['FEATURE_SOURCE_MATCHED'] = (
    base_label['FEATURE_SOURCE_MATCHED'].fillna(False).astype(bool)
)
labelled=label_bank_quarters(base_label,outcome,qord(LAST_MODEL_ANCHOR_QUARTER))
assert len(labelled)==EXPECTED_FINANCIAL_ROWS
assert not labelled.loc[~labelled.FEATURE_SOURCE_MATCHED,'FEATURE_SOURCE_MATCHED'].any()
print(labelled['LABEL_STATUS'].value_counts().to_string())
print('Confirmed positive bank-quarter labels:',int((labelled.fail_within_4q==1).sum()))

Synthetic target/exit/embargo edge cases: 9/9 PASS
LABEL_STATUS
OBSERVED_NO_FAILURE_4Q               1583330
UNKNOWN_CENSORED_OR_UNVERIFIED         64508
INSUFFICIENT_FOLLOWUP_AT_DATA_END      17529
CONFIRMED_FAILURE_WITHIN_4Q            10559
AMBIGUOUS_FAILURE_EVENT_LINK            6690
Confirmed positive bank-quarter labels: 10559


## 6 · Pre-registered temporal split; Test must include ≥50 **distinct** confirmed failing banks

The gap periods and 2015+ are explicitly **not** part of the registered model split. If Test has fewer than 50 unique failures, notebook **stops**; do not quietly change the test window after observing model performance. Do not use Validation or Test for data-dependent feature selection, clipping thresholds, imputation statistics, scaling, class balancing, hyperparameters, or the alarm threshold.

In [ ]:
labelled['SPLIT']='OUTSIDE_PRE_REGISTERED_SPLIT'
for s,(start,end) in BOUNDS.items():
    mask=labelled.Q_ORD.between(start,end,inclusive='both')
    labelled.loc[mask,'SPLIT']=s

labelled['MODEL_ELIGIBLE']=(labelled['SPLIT'].isin(BOUNDS) &
                            labelled['FEATURE_SOURCE_MATCHED'] &
                            labelled['fail_within_4q'].notna())

summary_rows=[]
for name in ['TRAIN','VALIDATION','TEST']:
    d=labelled[(labelled.SPLIT==name)&labelled.MODEL_ELIGIBLE]
    pos=d[d.fail_within_4q==1]
    summary_rows.append({'split':name,'rows':len(d),'positive_bank_quarters':len(pos),
        'negative_bank_quarters':int((d.fail_within_4q==0).sum()),
        'distinct_failed_banks':int(pos.CERT.nunique()),
        'distinct_banks':int(d.CERT.nunique()),
        'positive_share':float(len(pos)/len(d)) if len(d) else None,
        'start_q':SPLIT_BOUNDS[name][0],'end_q':SPLIT_BOUNDS[name][1]})
split_summary=pd.DataFrame(summary_rows)
display(split_summary)
assert int(split_summary.loc[split_summary.split=='TEST','distinct_failed_banks'].iloc[0])>=50, (
    'Course requirement: test period must contain >=50 distinct confirmed failed banks.')
assert all(split_summary.rows>0)
assert labelled.loc[labelled.MODEL_ELIGIBLE,'fail_within_4q'].notna().all()
assert labelled.loc[labelled.MODEL_ELIGIBLE,'FEATURE_SOURCE_MATCHED'].all()
assert not set(labelled.loc[labelled.SPLIT=='TRAIN','REPDTE']) & set(labelled.loc[labelled.SPLIT=='TEST','REPDTE'])
print('Temporal split AND minimum test failures: PASS')

,split,rows,positive_bank_quarters,negative_bank_quarters,distinct_failed_banks,distinct_banks,positive_share,start_q,end_q
0,TRAIN,1084227,8278,1075949,2191,22203,0.007635,1984Q1,2005Q4
1,VALIDATION,158131,1791,156340,465,8690,0.011326,2007Q1,2011Q4
2,TEST,266527,254,266273,78,6911,0.000953,2013Q1,2025Q2


Temporal split AND minimum test failures: PASS


## 7 · Anti-leakage tests and research-audit tables

`REPDTE`, `CERT`, `Q_ORD`, `SPLIT`, classification/status, and all event dates are **metadata**; they are NOT included in `FEATURE_COLUMNS`. We check that each anchor's feature source comes exclusively from an earlier calendar quarter, whether or not intermediate reports exist. A lagged feature is null if there is no report at the exact expected quarter. Historical coverage and missingness are profiled by split; **model feature eligibility is decided using Train only** in the optional preprocessing stage.

In [ ]:
FEATURE_COLUMNS=list(SELECTABLE) # pre-registered candidates; train-only filtering later
METADATA_COLS=['CERT','REPDTE','Q_ORD','FEATURE_SOURCE_Q_ORD',
               'FEATURE_SOURCE_MATCHED','SPLIT','MODEL_ELIGIBLE',
               'fail_within_4q','LABEL_STATUS']
assert not (set(FEATURE_COLUMNS) & (set(outcome.columns)|set(METADATA_COLS)|BAD_FEATURE_INPUTS))
assert not any(x.startswith(('HISTORY_','EVENT_','FAILDATE')) for x in FEATURE_COLUMNS)
assert (labelled.FEATURE_SOURCE_Q_ORD==labelled.Q_ORD-FEATURE_LAG_QUARTERS).all()
assert labelled.loc[labelled.FEATURE_SOURCE_MATCHED,'FEATURE_SOURCE_Q_ORD'].lt(
    labelled.loc[labelled.FEATURE_SOURCE_MATCHED,'Q_ORD']).all()
assert labelled[['CERT','REPDTE']].duplicated().sum()==0

# Strict 'no 0 for censored merger in horizon' test.
mergers_in_horizon=(labelled.REVISED_EXIT_CLASS_HISTORY.eq('MERGER_VERIFIED_HISTORY') &
    labelled.MERGER_Q.notna() & (labelled.MERGER_Q>labelled.Q_ORD) &
    (labelled.MERGER_Q<=labelled.Q_ORD+4))
assert not (labelled.loc[mergers_in_horizon,'fail_within_4q']==0).any()
assert not (labelled.loc[labelled.Q_ORD>qord(LAST_MODEL_ANCHOR_QUARTER),'fail_within_4q'].notna()).any()

# Only train observations determine feature-coverage diagnostics and later preprocessing.
train = labelled[(labelled.SPLIT=='TRAIN') & labelled.MODEL_ELIGIBLE]
assert not train.empty
coverage=[]
for f in FEATURE_COLUMNS:
    for name in ('TRAIN','VALIDATION','TEST'):
        part=labelled[(labelled.SPLIT==name)&labelled.MODEL_ELIGIBLE]
        coverage.append({'feature':f,'split':name,'rows':len(part),
                         'nonmissing_n':int(part[f].notna().sum()),
                         'coverage':float(part[f].notna().mean()) if len(part) else np.nan})
coverage_df=pd.DataFrame(coverage)
print('Low-coverage candidate features on Train:')
display(coverage_df[coverage_df.split=='TRAIN'].sort_values('coverage').head(12))

checks={
    'INPUT_43_YEAR_PARTITIONS':len(files)==43,
    'ALL_FINANCIAL_ROWS_PRESERVED':len(labelled)==EXPECTED_FINANCIAL_ROWS,
    'UNIQUE_BANK_QUARTER_KEYS':not labelled[['CERT','REPDTE']].duplicated().any(),
    'ONLY_PRIOR_QUARTER_FEATURES':bool((labelled.FEATURE_SOURCE_Q_ORD<labelled.Q_ORD).all()),
    'NO_OUTCOME_FIELDS_IN_PREDICTORS':not (set(FEATURE_COLUMNS)&set(outcome.columns)),
    'FOUR_QUARTER_EMBARGO_TRAIN_VAL':BOUNDS['TRAIN'][1]+5<=BOUNDS['VALIDATION'][0],
    'FOUR_QUARTER_EMBARGO_DEV_TEST':BOUNDS['VALIDATION'][1]+5<=BOUNDS['TEST'][0],
    'NO_IMMATURE_LABELS':labelled.loc[labelled.Q_ORD>qord(LAST_MODEL_ANCHOR_QUARTER),'fail_within_4q'].isna().all(),
    'UNKNOWN_MERGERS_NOT_NEGATIVE':not (labelled.loc[mergers_in_horizon,'fail_within_4q']==0).any(),
    'TEST_HAS_50_DISTINCT_FAILURES':int(split_summary.loc[split_summary.split=='TEST','distinct_failed_banks'].iloc[0])>=50,
    'TRAIN_VALIDATION_TEST_NONEMPTY':all(split_summary.rows>0),
}
gates_df=pd.DataFrame([{'gate':k,'passed':bool(v)} for k,v in checks.items()])
display(gates_df)
assert gates_df.passed.all(), 'At least one required acceptance gate failed.'
print('PART 4 Stage 1:',len(gates_df),'/',len(gates_df),'GATES PASS')

Low-coverage candidate features on Train:


,feature,split,rows,nonmissing_n,coverage
54,F_CRE_LOAN_SHARE,TRAIN,1084227,545319,0.502956
6,F_TIER1_RBC_PCT,TRAIN,1084227,683233,0.630157
72,F_NPL_CHANGE_4Q,TRAIN,1084227,986269,0.909652
69,F_ROA_CHANGE_4Q,TRAIN,1084227,996738,0.919307
18,F_NET_CHARGEOFFS_LOANS,TRAIN,1084227,1002057,0.924213
12,F_NONACCRUAL_ASSET,TRAIN,1084227,1011660,0.933070
15,F_PASTDUE90_ASSET,TRAIN,1084227,1011660,0.933070
75,F_ROA_ROLL4,TRAIN,1084227,1018265,0.939162
66,F_DEP_GROWTH_QOQ,TRAIN,1084227,1056237,0.974184
45,F_UNINSURED_SHARE,TRAIN,1084227,1060031,0.977684


,gate,passed
0,INPUT_43_YEAR_PARTITIONS,True
1,ALL_FINANCIAL_ROWS_PRESERVED,True
2,UNIQUE_BANK_QUARTER_KEYS,True
3,ONLY_PRIOR_QUARTER_FEATURES,True
4,NO_OUTCOME_FIELDS_IN_PREDICTORS,True
5,FOUR_QUARTER_EMBARGO_TRAIN_VAL,True
6,FOUR_QUARTER_EMBARGO_DEV_TEST,True
7,NO_IMMATURE_LABELS,True
8,UNKNOWN_MERGERS_NOT_NEGATIVE,True
9,TEST_HAS_50_DISTINCT_FAILURES,True


PART 4 Stage 1: 11 / 11 GATES PASS


## 8 · Export label audit, feature table and split files (no train fitting yet)

The eligible model tables have only key/time metadata, candidate numeric features and the target. *Audit metadata* with retrospective event dates and exit statuses is stored **separately** in `outputs/part4/` and excluded from train/val/test predictor tables. There are no raw FDIC datasets in the submission ZIP.

In [ ]:
def atomic_write_parquet(df, dest):
    dest=Path(dest)
    dest.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='fdic_part4_') as td:
        tmp=Path(td)/'data.parquet'
        df.to_parquet(tmp,index=False,engine='pyarrow',compression='zstd')
        part=dest.with_suffix('.parquet.partial')
        shutil.copy2(tmp,part)
        part.replace(dest)

# Keep eligible bank-quarters only; do not export 1.68M classified events in the notebook ZIP.
SAFE_OUTPUT=METADATA_COLS+[f for f in FEATURE_COLUMNS if f not in METADATA_COLS]
assert not (set(SAFE_OUTPUT) & BAD_FEATURE_INPUTS)
for name in ('TRAIN','VALIDATION','TEST'):
    part=labelled.loc[(labelled.SPLIT==name)&labelled.MODEL_ELIGIBLE,SAFE_OUTPUT].copy()
    part['fail_within_4q']=part['fail_within_4q'].astype('int8')
    atomic_write_parquet(part,MODEL_DATA/f'{name.lower()}_features_labels.parquet')
    print(name,'->',len(part),'rows')

# Audit report with all bank-quarter label statuses, including excluded ones.
label_audit=labelled.groupby(['SPLIT','LABEL_STATUS'],dropna=False).size().rename('rows').reset_index()
label_audit.to_csv(OUT/'label_status_by_split.csv',index=False)
feature_catalog.to_csv(OUT/'candidate_feature_catalog.csv',index=False)
coverage_df.to_csv(OUT/'feature_coverage_by_split.csv',index=False)
split_summary.to_csv(OUT/'split_counts_and_failures.csv',index=False)
gates_df.to_csv(OUT/'stage1_acceptance_gates.csv',index=False)

manifest={
 'project':'FDIC_BANK_FAILURE_PART4_V1','created_utc':datetime.now(timezone.utc).isoformat(),
 'input_clean_panel':'data_processed/clean_panel_v1',
 'input_history_exit':'data_processed/part2_history_exit_reference_v1/bank_exit_outcomes_with_history.parquet',
 'input_failure_alignment':'data_processed/part2_event_reference_v1/failure_event_alignment.parquet',
 'rows_total':int(len(labelled)), 'feature_lag_quarters':FEATURE_LAG_QUARTERS,
 'model_end':'2025Q2', 'label_horizon_quarters':4,
 'split_bounds':SPLIT_BOUNDS,'summary':split_summary.to_dict(orient='records'),
 'feature_candidates':FEATURE_COLUMNS,'gate_results':checks,
 'important_limitation':'report vintage/revisions unavailable; q-1 lag is an assumption, not a proven filing publication date',
 'label_policy':'aligned FDIC failures only; unknown/censored exits and ambiguous multiple CERT events not silently labeled negative',
 'unit_note':'financial amounts reported in FDIC source scale; ratios except *_PCT expressed as fractions',
}
(OUT/'stage1_manifest.json').write_text(json.dumps(manifest,indent=2,default=str,ensure_ascii=False),encoding='utf-8')
print('Saved stage1 reports:',OUT)
print('Saved versioned feature tables:',MODEL_DATA)

TRAIN -> 1084227 rows
VALIDATION -> 158131 rows
TEST -> 266527 rows
Saved stage1 reports: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part4/feature_pipeline_stage1_v1
Saved versioned feature tables: /content/drive/MyDrive/FDIC_Bank_Failure_Project/data_processed/feature_panel_v1


## 9 · Optional: fit preprocessing on **TRAIN ONLY** (never Test)

This is the **Stage 2** step of Part 4. Initially leave `RUN_PREPROCESSOR=False`; inspect `stage1_acceptance_gates.csv` and `split_counts_and_failures.csv` first. Once accepted, change to True and rerun **only this final cell** if its variables are still in memory. Fit decisions made **only using Train**: (a) discard features with less than 20% Train coverage, (b) median imputation (with missingness indicators), (c) robust scaling. Training-only percentile clipping may be added in a later, versioned sensitivity analysis; it is deliberately **not** performed in this initial reproducible pipeline. Save a `joblib` pipeline and metadata. No feature selection by test correlation, and no outcome/exit fields as predictors. **Do not resample test**.

This creates pipeline configuration, not a trained bank-failure classifier. Part 5 uses these fixed features and evaluates the Test period once.

In [ ]:

# ============================================================
# SECTION 9 — FIT PREPROCESSING ON TRAIN ONLY
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
import joblib
import numpy as np
import json

if RUN_PREPROCESSOR:

    # 1. Select features using TRAIN coverage only
    train_cov = (
        coverage_df.loc[coverage_df["split"] == "TRAIN"]
        .set_index("feature")["coverage"]
    )

    FIT_FEATURES = [
        f for f in FEATURE_COLUMNS
        if train_cov[f] >= 0.20
    ]

    assert len(FIT_FEATURES) >= 5
    assert "fail_within_4q" not in FIT_FEATURES

    # 2. Prepare TRAIN features
    X_train = (
        train[FIT_FEATURES]
        .astype("float64")
        .replace([np.inf, -np.inf], np.nan)
    )

    assert len(X_train) > 0
    assert X_train.columns.tolist() == FIT_FEATURES

    # 3. Define preprocessing pipeline
    prep = Pipeline([
        (
            "impute_train_medians",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        (
            "robust_scale",
            RobustScaler()
        )
    ])

    # CRITICAL: FIT ONLY ON TRAIN
    prep.fit(X_train)

    # 4. Validate fitted pipeline
    x_check = prep.transform(X_train.head(100))

    assert np.isfinite(x_check).all()

    FEATURE_NAMES_OUT = list(
        prep.get_feature_names_out(FIT_FEATURES)
    )

    assert x_check.shape[1] == len(FEATURE_NAMES_OUT)
    assert len(FEATURE_NAMES_OUT) == len(set(FEATURE_NAMES_OUT))

    # 5. Save fitted pipeline
    prep_path = OUT / "train_only_preprocessor.joblib"

    joblib.dump(
        {
            "pipeline": prep,
            "features": FIT_FEATURES,
            "feature_names_out": FEATURE_NAMES_OUT
        },
        prep_path
    )

    # 6. Save preprocessing manifest
    prep_manifest = {
        "fit_split": "TRAIN_ONLY",
        "train_rows_used": int(len(X_train)),
        "selected_features": FIT_FEATURES,
        "features_before_transform": len(FIT_FEATURES),
        "features_after_transform": len(FEATURE_NAMES_OUT),
        "selection_rule": "Train coverage >= 20%",
        "imputation": "TRAIN median + missing indicators",
        "scaling": "RobustScaler fitted on TRAIN",
        "clipping": "None",
        "test_not_used_for_fitting": True,
        "validation_not_used_for_fitting": True
    }

    (OUT / "preprocessing_manifest.json").write_text(
        json.dumps(
            prep_manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("SECTION 9 — PASS")
    print("Train rows:", f"{len(X_train):,}")
    print("Selected features:", len(FIT_FEATURES))
    print("Output features:", len(FEATURE_NAMES_OUT))
    print("Pipeline saved:", prep_path)

else:
    print("SECTION 9 SKIPPED")
    print("Set RUN_PREPROCESSOR=True before running.")

SECTION 9 — PASS
Train rows: 1,084,227
Selected features: 26
Output features: 48
Pipeline saved: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part4/feature_pipeline_stage1_v1/train_only_preprocessor.joblib


In [ ]:

# ============================================================
# SECTION 9B — TRANSFORM ALL SPLITS + FINAL REPORT ZIP
# ============================================================

import json
import zipfile
import joblib
import numpy as np
import pandas as pd

if RUN_PREPROCESSOR:

    # 1. Load TRAIN-fitted preprocessing
    prep_path = OUT / "train_only_preprocessor.joblib"
    manifest_path = OUT / "preprocessing_manifest.json"

    assert prep_path.is_file()
    assert manifest_path.is_file()

    bundle = joblib.load(prep_path)
    pipeline = bundle["pipeline"]

    selected_features = bundle["features"]
    feature_names = bundle["feature_names_out"]

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    # 2. Verify TRAIN-only fitting policy
    assert manifest["fit_split"] == "TRAIN_ONLY"
    assert manifest["test_not_used_for_fitting"] is True
    assert manifest["validation_not_used_for_fitting"] is True
    assert selected_features == manifest["selected_features"]

    # 3. Define output directory
    PROCESSED_DIR = MODEL_DATA / "preprocessed_v1"
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    validation_reports = []

    # 4. Transform each split without refitting
    for split in ["TRAIN", "VALIDATION", "TEST"]:

        input_file = (
            MODEL_DATA
            / f"{split.lower()}_features_labels.parquet"
        )

        assert input_file.is_file(), f"Missing {input_file}"

        df = pd.read_parquet(input_file)

        # Preserve identifiers and target separately
        metadata_cols = [
            "CERT",
            "REPDTE",
            "Q_ORD",
            "fail_within_4q"
        ]

        assert all(c in df.columns for c in metadata_cols)
        assert all(c in df.columns for c in selected_features)

        metadata = (
            df[metadata_cols]
            .reset_index(drop=True)
            .copy()
        )

        assert metadata["fail_within_4q"].notna().all()
        assert metadata["fail_within_4q"].isin([0, 1]).all()
        assert not metadata[["CERT", "REPDTE"]].duplicated().any()

        # Same feature order for all three splits
        X = (
            df[selected_features]
            .astype("float64")
            .replace([np.inf, -np.inf], np.nan)
        )

        # IMPORTANT: TRANSFORM ONLY
        # Never call fit() or fit_transform() here
        X_transformed = pipeline.transform(X)

        if hasattr(X_transformed, "toarray"):
            X_transformed = X_transformed.toarray()

        assert X_transformed.shape == (
            len(df), len(feature_names)
        )
        assert np.isfinite(X_transformed).all()

        X_ready = pd.DataFrame(
            X_transformed,
            columns=feature_names
        )

        assert X_ready.columns.is_unique
        assert X_ready.isna().sum().sum() == 0

        # 5. Combine metadata with transformed features
        output = pd.concat(
            [metadata, X_ready],
            axis=1
        )

        # Check predictors do not contain identifiers/target
        assert not (
            set(feature_names) & set(metadata_cols)
        )

        output_file = (
            PROCESSED_DIR
            / f"{split.lower()}_model_ready.parquet"
        )

        output.to_parquet(
            output_file,
            index=False,
            compression="zstd"
        )

        # 6. Validation report
        positive_banks = int(
            metadata.loc[
                metadata["fail_within_4q"] == 1,
                "CERT"
            ].nunique()
        )

        validation_reports.append({
            "split": split,
            "rows": len(df),
            "features_before": len(selected_features),
            "features_after": len(feature_names),
            "missing_after": int(X_ready.isna().sum().sum()),
            "infinite_after": int(
                np.isinf(X_transformed).sum()
            ),
            "distinct_positive_banks": positive_banks,
            "status": "PASS",
            "output_file": str(output_file)
        })

        print(
            f"{split}: {len(df):,} rows | "
            f"{len(feature_names)} features | PASS"
        )

    # 7. Validate all three splits
    validation_df = pd.DataFrame(validation_reports)

    assert len(validation_df) == 3
    assert (validation_df["missing_after"] == 0).all()
    assert (validation_df["infinite_after"] == 0).all()
    assert validation_df["features_after"].nunique() == 1

    test_positive_banks = int(
        validation_df.loc[
            validation_df["split"] == "TEST",
            "distinct_positive_banks"
        ].iloc[0]
    )

    assert test_positive_banks >= 50, (
        f"TEST has only {test_positive_banks} "
        "distinct failing banks."
    )

    # 8. Save Stage 2 validation report
    validation_path = (
        OUT / "stage2_preprocessing_validation.csv"
    )

    validation_df.to_csv(
        validation_path,
        index=False
    )

    print("\nSECTION 9B — PASS")
    display(validation_df)

else:
    print("SECTION 9B — Preprocessing skipped.")

# ============================================================
# FINAL ZIP — AFTER SECTION 9B
# Includes Stage 1 + Stage 2 reports
# ============================================================

zip_path = (
    OUT / "FDIC_Part4_Stage1_Label_Feature_Split_Reports.zip"
)

report_files = [
    p for p in sorted(OUT.iterdir())
    if p.is_file()
    and p.suffix.lower() in {".csv", ".json"}
]

if RUN_PREPROCESSOR:
    required_stage2 = {
        "preprocessing_manifest.json",
        "stage2_preprocessing_validation.csv"
    }
    assert required_stage2.issubset(
        {p.name for p in report_files}
    )

# Store lightweight reports, not large bank-level Parquet
with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for path in report_files:
        archive.write(path, arcname=path.name)

# Check ZIP contents
with zipfile.ZipFile(zip_path, "r") as archive:
    archived_names = set(archive.namelist())
    assert all(p.name in archived_names for p in report_files)
    assert archive.testzip() is None

print("\nFINAL PART 4 REPORT ZIP CREATED")
print("Reports included:", len(report_files))
print("ZIP location:", zip_path)
print("Model-ready data:", MODEL_DATA / "preprocessed_v1")


TRAIN: 1,084,227 rows | 48 features | PASS
VALIDATION: 158,131 rows | 48 features | PASS
TEST: 266,527 rows | 48 features | PASS

SECTION 9B — PASS


,split,rows,features_before,features_after,missing_after,infinite_after,distinct_positive_banks,status,output_file
0,TRAIN,1084227,26,48,0,0,2191,PASS,/content/drive/MyDrive/FDIC_Bank_Failure_Proje...
1,VALIDATION,158131,26,48,0,0,465,PASS,/content/drive/MyDrive/FDIC_Bank_Failure_Proje...
2,TEST,266527,26,48,0,0,78,PASS,/content/drive/MyDrive/FDIC_Bank_Failure_Proje...



FINAL PART 4 REPORT ZIP CREATED
Reports included: 8
ZIP location: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part4/feature_pipeline_stage1_v1/FDIC_Part4_Stage1_Label_Feature_Split_Reports.zip
Model-ready data: /content/drive/MyDrive/FDIC_Bank_Failure_Project/data_processed/feature_panel_v1/preprocessed_v1


## 10 · Interpretation & next steps

**Review in this order:**

1. `stage1_acceptance_gates.csv` must be all PASS; `split_counts_and_failures.csv` must show ≥50 unique confirmed failures in TEST (not simply ≥50 positive rows).
2. `label_status_by_split.csv` identifies positive / negative / censored / review observations; these cannot be collapsed into a simple nonfailure class.
3. `feature_coverage_by_split.csv` quantifies changing historical availability (e.g. uninsured deposits, regulatory capital). Candidate inclusion is **theory-led**, and data-dependent availability selection uses **Train only**.
4. Inspect the gap periods and feature availability before `RUN_PREPROCESSOR=True` and before modeling. If Train-only preprocessing is fitted, the exact `train_only_preprocessor.joblib` is reused in Part 5 for Validation and Test (never refit).
5. **Do not run a model on Test until the Part 5 model, hyperparameters, validation scheme and cost-based alarm threshold are frozen.**

**Limits disclosed upfront:** This exercise tests FDIC-failure prediction, not every distressed merger; assisted mergers are not automatically failures. Study ends 2026Q2 and labels are limited through 2025Q2. The pre-registered 2013Q1–2025Q2 Test is a long out-of-time holdout; it covers recent periods but contains major structural change; annual cohort monitoring and sensitivity checks remain important. The 2023–2025 observations are part of the same held-out Test, not an additional tuning or validation set. FDIC financial data may contain retrospectively revised values; a 1Q lag is not proof of actual publication vintage. Do not claim robust out-of-time performance until Part 5 is evaluated.

**References:** See links in Section 0. The reporting instructions require academic citations for sources, code and AI assistance.